# DexGrasp training on Kaggle

Enable **GPU** and **Internet** in Kaggle before running. Run the cells from top to bottom.

The W&B key is written directly in this notebook as requested. Do not push the notebook after replacing the placeholder with a real key.

## 1. Clone repository and switch branch

In [ ]:
!git clone https://github.com/hunghehe2205/mjlab.git /kaggle/working/mjlab
%cd /kaggle/working/mjlab
!git switch DexGrasp
!git pull --ff-only origin DexGrasp
!git log -1 --oneline

## 2. Install uv and project dependencies

In [ ]:
!mkdir -p /kaggle/working/bin
!curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR=/kaggle/working/bin sh
!/kaggle/working/bin/uv sync --frozen --no-dev --extra cu128
!/kaggle/working/bin/uv run --extra cu128 python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"
!nvidia-smi

## 3. Log in to W&B

Replace the placeholder below with your real API key. `WANDB_USERNAME` is optional; uncomment it when logging to a team entity.

In [ ]:
import os

os.environ["WANDB_API_KEY"] = "PASTE_YOUR_WANDB_API_KEY_HERE"
# os.environ["WANDB_USERNAME"] = "your-user-or-team"
os.environ["WANDB_CONSOLE"] = "wrap"

In [ ]:
!/kaggle/working/bin/uv run --extra cu128 wandb login --relogin "$WANDB_API_KEY"

## 4. Smoke train

Run this first. Check on W&B that `Episode_Termination/nan` stays at zero.

In [ ]:
!/kaggle/working/bin/uv run --extra cu128 train Mjlab-DexGrasp-UR5eRH5DG2 \
  --gpu-ids "[0]" \
  --env.scene.num-envs 8 \
  --agent.num-steps-per-env 8 \
  --agent.max-iterations 20 \
  --agent.save-interval 10 \
  --agent.run-name dexgrasp_teacher_kaggle_smoke \
  --agent.logger wandb \
  --agent.wandb-project mjlab-dexgrasp \
  --agent.wandb-tags "('kaggle', 'dexgrasp', 'smoke', 'video')" \
  --agent.upload-model True \
  --video True \
  --video-length 100 \
  --video-interval 100 \
  --enable-nan-guard True \
  --env.sim.nan-guard.output-dir /kaggle/working/nan_dumps \
  --log-root /kaggle/working/logs

## 5. Full train

The full run uses 4,096 environments on one T4 for 10,000 iterations. Reduce `num-envs` to 2,048 if the GPU runs out of memory. A 200-frame video is uploaded to W&B every 20,000 environment steps.

In [ ]:
!/kaggle/working/bin/uv run --extra cu128 train Mjlab-DexGrasp-UR5eRH5DG2 \
  --gpu-ids "[0]" \
  --env.scene.num-envs 4096 \
  --agent.num-steps-per-env 70 \
  --agent.max-iterations 10000 \
  --agent.save-interval 100 \
  --agent.run-name dexgrasp_teacher_kaggle_4096env_1xT4 \
  --agent.logger wandb \
  --agent.wandb-project mjlab-dexgrasp \
  --agent.wandb-tags "('kaggle', 'dexgrasp', 'full', '4096env', '1xT4', 'video')" \
  --agent.upload-model True \
  --video True \
  --video-length 200 \
  --video-interval 20000 \
  --enable-nan-guard True \
  --env.sim.nan-guard.output-dir /kaggle/working/nan_dumps \
  --log-root /kaggle/working/logs

W&B automatically receives training metrics, videos under Media, resolved environment/agent config, git commit and diff, and model checkpoints. NaN dumps remain in `/kaggle/working/nan_dumps` for manual download when needed.